# Engine PINN — Self-Run Notebook (Standalone)

This notebook is **fully self-contained**:
- No `engine_pinn` package imports required.
- You only need:
  1) this `.ipynb` file
  2) an optional CSV data file

If CSV is missing, synthetic data is generated automatically.


In [ ]:
# Optional dependency bootstrap (safe for Colab/local)
import importlib
import subprocess
import sys

REQUIRED = ["numpy", "pandas", "torch", "sklearn", "matplotlib", "seaborn"]
MISSING = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except Exception:
        MISSING.append(pkg)

if MISSING:
    print("Installing missing packages:", MISSING)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *MISSING])
else:
    print("All required packages already available.")


In [ ]:
import logging
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set(style="whitegrid")


## 1) Config and reproducibility


In [ ]:
@dataclass
class PhysicsConfig:
    lhv: float = 42000.0
    nox_b_init: float = 5.0

@dataclass
class TrainConfig:
    seed: int = 42
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-5
    lambda_phys: float = 1.0
    lambda_mono: float = 0.2
    epochs_phase1: int = 25
    epochs_phase2: int = 50
    patience: int = 15
    test_size: float = 0.15
    val_size: float = 0.15
    data_csv: Path = Path("engine_data.csv")  # Place your CSV next to notebook or set full path
    artifacts_dir: Path = Path("artifacts")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("selfrun-notebook")

train_cfg = TrainConfig()
phys_cfg = PhysicsConfig()
set_seed(train_cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2) Data utilities (embedded in notebook)


In [ ]:
FEATURE_COLUMNS = ["BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda"]
TARGET_COLUMNS = ["BSFC", "NOx"]

class EngineDataset(Dataset):
    def __init__(self, x_scaled: np.ndarray, y_scaled: np.ndarray, x_raw: np.ndarray, y_raw: np.ndarray) -> None:
        if len(x_scaled) != len(y_scaled):
            raise ValueError("x and y lengths must match")
        self.x = torch.tensor(x_scaled, dtype=torch.float32)
        self.y = torch.tensor(y_scaled, dtype=torch.float32)
        self.x_raw = torch.tensor(x_raw, dtype=torch.float32)
        self.y_raw = torch.tensor(y_raw, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {"x": self.x[idx], "y": self.y[idx], "x_raw": self.x_raw[idx], "y_raw": self.y_raw[idx]}


def load_or_generate_dataframe(csv_path: Path, n_samples: int = 1500, random_state: int = 42) -> pd.DataFrame:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        required = set(FEATURE_COLUMNS + TARGET_COLUMNS)
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f"CSV missing columns: {sorted(missing)}")
        return df

    rng = np.random.default_rng(random_state)
    bmep = rng.uniform(2.0, 18.0, n_samples)
    h2 = rng.uniform(0.0, 40.0, n_samples)
    spark = rng.uniform(-10.0, 30.0, n_samples)
    lamb = rng.uniform(0.85, 1.25, n_samples)

    fmep = 0.4 + 0.08 * bmep + 0.02 * rng.normal(size=n_samples)
    eff = np.clip(0.28 + 0.015 * bmep - 0.0015 * (spark - 10) ** 2 + 0.003 * h2, 0.2, 0.8)
    temp = 800 + 9.0 * spark + 6.5 * bmep + 2.0 * h2 + 20 * rng.normal(size=n_samples)
    temp = np.clip(temp, 300, None)

    bsfc = (3600.0 / 42000.0) * ((bmep + fmep) / (np.maximum(bmep, 1e-3) * eff))
    bsfc *= 1.0 + 0.02 * rng.normal(size=n_samples)

    a_true, b_true = 1200.0, 4.8
    nox = a_true * np.exp(-b_true / np.maximum(temp / 1000.0, 1e-4))
    nox *= 1 + 0.06 * np.maximum(spark, 0) / 30.0
    nox *= 1.0 + 0.03 * rng.normal(size=n_samples)
    nox = np.clip(nox, 1e-3, None)

    return pd.DataFrame({
        "BMEP": bmep,
        "H2_percentage": h2,
        "Spark_Ignition_Timing": spark,
        "Lambda": lamb,
        "BSFC": bsfc,
        "NOx": nox,
    })


def build_dataloaders(df: pd.DataFrame, batch_size: int, test_size: float, val_size: float, seed: int):
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=seed)
    val_ratio = val_size / (1 - test_size)
    train_df, val_df = train_test_split(train_df, test_size=val_ratio, random_state=seed)

    x_scaler, y_scaler = StandardScaler(), StandardScaler()

    x_train = x_scaler.fit_transform(train_df[FEATURE_COLUMNS])
    y_train = y_scaler.fit_transform(train_df[TARGET_COLUMNS])

    x_val = x_scaler.transform(val_df[FEATURE_COLUMNS])
    y_val = y_scaler.transform(val_df[TARGET_COLUMNS])

    x_test = x_scaler.transform(test_df[FEATURE_COLUMNS])
    y_test = y_scaler.transform(test_df[TARGET_COLUMNS])

    train_ds = EngineDataset(x_train, y_train, train_df[FEATURE_COLUMNS].to_numpy(), train_df[TARGET_COLUMNS].to_numpy())
    val_ds = EngineDataset(x_val, y_val, val_df[FEATURE_COLUMNS].to_numpy(), val_df[TARGET_COLUMNS].to_numpy())
    test_ds = EngineDataset(x_test, y_test, test_df[FEATURE_COLUMNS].to_numpy(), test_df[TARGET_COLUMNS].to_numpy())

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False),
        x_scaler,
        y_scaler,
    )


## 3) PINN model (embedded)


In [ ]:
class EfficiencyActivation(nn.Module):
    def __init__(self, low: float = 0.2, high: float = 0.8) -> None:
        super().__init__()
        if high <= low:
            raise ValueError("high must be > low")
        self.low = low
        self.scale = high - low

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.low + self.scale * torch.sigmoid(x)


class LatentActivationBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.softplus = nn.Softplus()
        self.eff = EfficiencyActivation(0.2, 0.8)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        if z.shape[-1] != 3:
            raise ValueError("Expected 3 latent channels")
        l1 = self.softplus(z[:, 0:1])
        l2 = self.eff(z[:, 1:2])
        l3 = self.softplus(z[:, 2:3])
        return torch.cat([l1, l2, l3], dim=-1)


class EnginePINN(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dims: Tuple[int, int] = (64, 32), lhv: float = 42000.0,
                 nox_a_init: float = 500.0, nox_b_init: float = 5.0) -> None:
        super().__init__()
        h1, h2 = hidden_dims
        self.lhv = lhv

        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(),
        )
        self.latent_head = nn.Linear(h2, 3)
        self.latent_act = LatentActivationBlock()

        self.nox_a_raw = nn.Parameter(torch.tensor(float(max(nox_a_init, 1e-3))))
        self.nox_b_raw = nn.Parameter(torch.tensor(float(max(nox_b_init, 1e-3))))

        self._init_latent_biases()

    def _init_latent_biases(self) -> None:
        nn.init.xavier_uniform_(self.latent_head.weight)
        with torch.no_grad():
            self.latent_head.bias[0] = 0.5
            l2_target = (0.35 - 0.2) / 0.6
            self.latent_head.bias[1] = torch.log(torch.tensor(l2_target / (1 - l2_target)))
            self.latent_head.bias[2] = 1.0

    @property
    def nox_a(self) -> torch.Tensor:
        return torch.nn.functional.softplus(self.nox_a_raw)

    @property
    def nox_b(self) -> torch.Tensor:
        return torch.nn.functional.softplus(self.nox_b_raw)

    def set_physics_scalar_training(self, enabled: bool) -> None:
        self.nox_a_raw.requires_grad = enabled
        self.nox_b_raw.requires_grad = enabled

    def forward(self, x_scaled: torch.Tensor, x_raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        eps = 1e-6
        feats = self.feature_extractor(x_scaled)
        z = self.latent_act(self.latent_head(feats))

        bmep = x_raw[:, 0:1]
        l1, l2, l3 = z[:, 0:1], z[:, 1:2], z[:, 2:3]

        bsfc = (3600.0 / self.lhv) * ((bmep + l1) / (torch.clamp(bmep, min=eps) * torch.clamp(l2, min=eps)))
        nox = self.nox_a * torch.exp(-self.nox_b / torch.clamp(l3, min=eps))
        nox = torch.clamp(nox, min=eps)

        return {"bsfc": bsfc, "nox": nox, "latents": z}


## 4) Loss + Trainer (embedded)


In [ ]:
class PINNLoss(nn.Module):
    def __init__(self, lambda_phys: float = 1.0, lambda_mono: float = 0.2) -> None:
        super().__init__()
        self.lambda_phys = lambda_phys
        self.lambda_mono = lambda_mono
        self.mse = nn.MSELoss()

    def _physics_penalty(self, z: torch.Tensor) -> torch.Tensor:
        l1, l2, l3 = z[:, 0], z[:, 1], z[:, 2]
        return (
            torch.relu(-l1).pow(2).mean()
            + torch.relu(0.2 - l2).pow(2).mean()
            + torch.relu(l2 - 1.0).pow(2).mean()
            + torch.relu(-l3).pow(2).mean()
        )

    def _mono_penalty(self, x_raw: torch.Tensor, bsfc_pred: torch.Tensor, nox_pred: torch.Tensor) -> torch.Tensor:
        g_bsfc = torch.autograd.grad(bsfc_pred, x_raw, grad_outputs=torch.ones_like(bsfc_pred), create_graph=True, retain_graph=True)[0]
        g_nox = torch.autograd.grad(nox_pred, x_raw, grad_outputs=torch.ones_like(nox_pred), create_graph=True, retain_graph=True)[0]

        dbsfc_dbmep = g_bsfc[:, 0]
        dnox_dbmep = g_nox[:, 0]
        dnox_dspark = g_nox[:, 2]

        return (
            torch.relu(dbsfc_dbmep).pow(2).mean()
            + torch.relu(-dnox_dbmep).pow(2).mean()
            + torch.relu(-dnox_dspark).pow(2).mean()
        )

    def forward(self, out: Dict[str, torch.Tensor], y_raw: torch.Tensor, x_raw: torch.Tensor):
        eps = 1e-6
        bsfc_true = y_raw[:, 0:1]
        nox_true = torch.clamp(y_raw[:, 1:2], min=eps)

        data = self.mse(out["bsfc"], bsfc_true) + self.mse(torch.log(torch.clamp(out["nox"], min=eps)), torch.log(nox_true))
        phys = self._physics_penalty(out["latents"])
        mono = self._mono_penalty(x_raw, out["bsfc"], out["nox"])
        total = data + self.lambda_phys * phys + self.lambda_mono * mono
        return total, data, phys, mono


class Trainer:
    def __init__(self, model: nn.Module, loss_fn: PINNLoss, optimizer: torch.optim.Optimizer,
                 device: torch.device, checkpoint_dir: Path, patience: int = 15) -> None:
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=6)
        self.patience = patience

    def _step(self, batch: Dict[str, torch.Tensor], training: bool):
        x = batch["x"].to(self.device)
        y_raw = batch["y_raw"].to(self.device)
        x_raw = batch["x_raw"].to(self.device).detach().requires_grad_(True)

        if training:
            self.optimizer.zero_grad(set_to_none=True)

        out = self.model(x, x_raw)
        total, data, phys, mono = self.loss_fn(out, y_raw, x_raw)

        if training:
            total.backward()
            self.optimizer.step()

        return float(total.detach().cpu()), float(data.detach().cpu()), float(phys.detach().cpu()), float(mono.detach().cpu())

    def _run_epoch(self, loader: DataLoader, training: bool):
        self.model.train(training)
        vals = [self._step(batch, training) for batch in loader]
        arr = np.array(vals)
        return arr.mean(axis=0)

    def fit(self, train_loader: DataLoader, val_loader: DataLoader, epochs_phase1: int, epochs_phase2: int):
        best_val = float("inf")
        no_improve = 0
        history = {"train_total": [], "val_total": []}

        total_epochs = epochs_phase1 + epochs_phase2
        for epoch in range(total_epochs):
            phase = 1 if epoch < epochs_phase1 else 2
            self.model.set_physics_scalar_training(enabled=(phase == 2))

            tr_total, tr_data, tr_phys, tr_mono = self._run_epoch(train_loader, training=True)
            va_total, va_data, va_phys, va_mono = self._run_epoch(val_loader, training=False)

            history["train_total"].append(float(tr_total))
            history["val_total"].append(float(va_total))
            self.scheduler.step(float(va_total))

            logger.info(
                "Epoch %d/%d phase=%d train=%.6f val=%.6f data=%.6f phys=%.6f mono=%.6f",
                epoch + 1, total_epochs, phase, tr_total, va_total, va_data, va_phys, va_mono,
            )

            if va_total < best_val:
                best_val = va_total
                no_improve = 0
                torch.save(self.model.state_dict(), self.checkpoint_dir / "best_model.pt")
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    logger.info("Early stopping at epoch %d", epoch + 1)
                    break

        ckpt = self.checkpoint_dir / "best_model.pt"
        if ckpt.exists():
            self.model.load_state_dict(torch.load(ckpt, map_location=self.device))

        return history


## 5) Plot helpers + Evaluation (embedded)


In [ ]:
def plot_training_curves(history: Dict[str, List[float]], save_path: Path) -> None:
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_total"], label="Train Total")
    plt.plot(history["val_total"], label="Val Total")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_latent_trends(df: pd.DataFrame, save_path: Path) -> None:
    req = {"BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda", "L1_FMEP", "L2_Indicated_Efficiency", "L3_Temp_Potential"}
    missing = req.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey="row")
    latents = ["L1_FMEP", "L2_Indicated_Efficiency", "L3_Temp_Potential"]
    inputs = ["BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda"]

    for r, lat in enumerate(latents):
        for c, inp in enumerate(inputs):
            ax = axes[r, c]
            sns.regplot(data=df, x=inp, y=lat, ax=ax, scatter_kws={"s": 10, "alpha": 0.4}, line_kws={"color": "red"})
            ax.set_ylabel(lat if c == 0 else "")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)


def evaluate_and_collect_latents(model: EnginePINN, loader: DataLoader, y_scaler: StandardScaler, device: torch.device) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            x_raw = batch["x_raw"].to(device)
            out = model(x, x_raw)

            y_true = y_scaler.inverse_transform(batch["y"].cpu().numpy())
            y_pred = np.hstack([out["bsfc"].cpu().numpy(), out["nox"].cpu().numpy()])
            latent = out["latents"].cpu().numpy()
            x_raw_np = batch["x_raw"].cpu().numpy()

            for i in range(len(x_raw_np)):
                rows.append({
                    "BMEP": x_raw_np[i, 0],
                    "H2_percentage": x_raw_np[i, 1],
                    "Spark_Ignition_Timing": x_raw_np[i, 2],
                    "Lambda": x_raw_np[i, 3],
                    "BSFC_true": y_true[i, 0],
                    "NOx_true": y_true[i, 1],
                    "BSFC_pred": y_pred[i, 0],
                    "NOx_pred": y_pred[i, 1],
                    "L1_FMEP": latent[i, 0],
                    "L2_Indicated_Efficiency": latent[i, 1],
                    "L3_Temp_Potential": latent[i, 2],
                })
    return pd.DataFrame(rows)


## 6) Run the full pipeline


In [ ]:
# If your CSV is not in current folder, set an absolute path here:
# train_cfg.data_csv = Path('/content/engine_data.csv')

df = load_or_generate_dataframe(train_cfg.data_csv)
print("Data rows:", len(df))
print(df.head())

train_loader, val_loader, test_loader, x_scaler, y_scaler = build_dataloaders(
    df=df,
    batch_size=train_cfg.batch_size,
    test_size=train_cfg.test_size,
    val_size=train_cfg.val_size,
    seed=train_cfg.seed,
)

nox_a_init = max(float(df["NOx"].max()), 1e-3)
model = EnginePINN(lhv=phys_cfg.lhv, nox_a_init=nox_a_init, nox_b_init=phys_cfg.nox_b_init).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
loss_fn = PINNLoss(lambda_phys=train_cfg.lambda_phys, lambda_mono=train_cfg.lambda_mono)
trainer = Trainer(model=model, loss_fn=loss_fn, optimizer=optimizer, device=device,
                  checkpoint_dir=train_cfg.artifacts_dir / "checkpoints", patience=train_cfg.patience)

history = trainer.fit(train_loader, val_loader, epochs_phase1=train_cfg.epochs_phase1, epochs_phase2=train_cfg.epochs_phase2)

results_df = evaluate_and_collect_latents(model, test_loader, y_scaler, device)

train_cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)
training_curve_path = train_cfg.artifacts_dir / "training_curves.png"
latent_path = train_cfg.artifacts_dir / "latent_sensitivity.png"
results_path = train_cfg.artifacts_dir / "test_predictions_with_latents.csv"

plot_training_curves(history, training_curve_path)
plot_latent_trends(results_df, latent_path)
results_df.to_csv(results_path, index=False)

print("Saved:", training_curve_path)
print("Saved:", latent_path)
print("Saved:", results_path)


In [ ]:
results_df[["BSFC_true", "BSFC_pred", "NOx_true", "NOx_pred"]].describe().T
